# k5 — Stage 2: xếp hạng lại top-N (ColBERT BGE-M3 / cross-encoder) + bài nộp (GPU T4 ×2)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`, `mir2-runs` (có `qtrans/queries_*.parquet`).
**Baseline: `RUN = 'rrf9_w'`, `METHODS = ['cross']`** (`BAAI/bge-reranker-v2-m3`), chọn theo tập có nhãn 40 câu
(19/10): cross-encoder hơn rrf9 có ý nghĩa, ColBERT thì không; rrf9_w (en_lsr, en_bm25 trọng số 0,5) hơn rrf9. `TOP_N` phải lớn hơn
k_chunk (F2 tính theo TẬP: xếp hạng lại trong đúng tập nộp không đổi điểm). Dự án cũ: ColBERT top-400 ~100 phút / 2 GPU.
**Đầu ra:** `runs/<run>_<method>.parquet`, `submissions/*.zip` (doc_id số nguyên, tự giảm k_chunk nếu > 99 MB).
**Thử trên bộ nhãn:** thêm file danh sách câu (vd `queries.csv`) làm Input, đặt `QIDS = 'queries.csv'` → chỉ rerank các câu
đó, ra `runs/<run>_<method>_sub.parquet`, không tạo bài nộp. bge top-400 ~6 s/câu/GPU (đo 19/10).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k5-v10 (15/10/2026)'
RUN = 'rrf9_w'                     # baseline: rrf9_w + cross (đánh giá 19/10)
METHODS = ['cross']                  # thêm 'colbert' nếu muốn so sánh; [] = chỉ nộp RUN có sẵn
TOP_N = 400
K_DOC, K_CHUNK = 200, 185
QIDS = ''                            # thử trên bộ nhãn: 'queries.csv' (tìm trong /kaggle/input) hoặc '1,2,3'; '' = cả 1.200 câu
DOC_AGG = 'max'                      # thứ hạng tài liệu khi nộp: max (đoạn tốt nhất) | sum2 | sum3
MAX_PER_DOC = [0]                    # tối đa n đoạn mỗi tài liệu trong 185 đoạn; [0] = không giới hạn; [1, 2] = 2 bài nộp _mpd1, _mpd2
CHUNK_PROBS = ''                     # phân bổ lại ngân sách đoạn theo P(yes): tên run Qwen3 trong Input (vd 'lin07_qwen3i') → thêm bài nộp _cbud; '' = tắt
CHUNK_K_MIN = 100                    # số đoạn tối thiểu mỗi câu khi phân bổ (k tối đa 400)
CHUNK_CALIB = ''                     # hiệu chỉnh P(yes) theo 'precision,recall' đoạn trên LB của bài 185 đoạn (vd '0.0525,0.1555') → bài _cbudcal
CALIB_QIDS = 'public600_qids.txt'   # câu LB đã chấm (Input) để khớp hiệu chỉnh
REUSE = {}                           # dùng lại điểm đã chấm: {phương pháp: run hoặc [các run] trong Input}, vd {'cross': ['lin07_cross', 'lin12_07_cross'], 'qwen3i': 'lin07_qwen3i_1200'} → chỉ chấm đoạn mới
ENSEMBLE = False                     # True + METHODS có 'cross' và 'qwen3i' → thêm run <RUN>_ensi
SUBMIT_RUN = True                    # False: không nộp RUN chưa rerank (chỉ nộp run đã rerank)
CROSS_MODEL = ''                     # model bge đã fine-tune (thư mục có finetune_meta.json trong Input, vd 'bge') → run <RUN>_crossft
QWEN_LORA = ''                       # adapter LoRA Qwen3 đã fine-tune (vd 'qwen') → run <RUN>_qwen3ift
PUBMED_ONLY = False                  # True: chỉ chấm đoạn bài PubMed (model ft dùng riêng cho tiếng Anh) → run <RUN>_<method>ftpm, không ensemble / bài nộp
FT_ALPHA = 1.0                       # WiSE-FT: < 1 kéo model ft về gần model gốc (0,5 = nửa đường) → tên run ft50
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks', 'qtrans'))

In [ ]:
# (2) Xếp hạng lại trên 2 GPU (chia theo câu hỏi), ghi vào /kaggle/working/runs
NAMES = [x.strip() for x in RUN.split(',') if x.strip()]  # METHODS = [] → nộp được nhiều run có sẵn: 'a,b,c'
assert len(NAMES) == 1 or not METHODS, 'Nhiều RUN chỉ dùng với METHODS = [] (chỉ tạo bài nộp)'
RUN = NAMES[0]
hits = locate_all(f'runs/{RUN}.parquet') or locate_all('runs/vi_dense.parquet')   # bỏ qua runs/ của output k5 cũ
assert hits, (f'Không thấy runs/{RUN}.parquet hay 9 nhánh (runs/vi_dense.parquet…) trong /kaggle/input: '
              'Add Input mir2-runs (output k4)')
RUNS_IN = hits[0] / 'runs'
OUT = f'{K.work}/runs'
os.makedirs(OUT, exist_ok=True)
if QIDS and not os.path.exists(QIDS) and not QIDS.replace(',', '').isdigit():
    hits = sorted(glob.glob(f'{KAGGLE}/input/**/{QIDS}', recursive=True), key=len)
    assert hits, f'Không thấy {QIDS} trong /kaggle/input'
    QIDS = hits[0]
QARG = f' --qids "{QIDS}"' if QIDS else ''        # run thử trên bộ nhãn tên <run>_<method>_sub
print('câu hỏi:', QIDS or 'cả 1.200 câu')
if not os.path.exists(f'{RUNS_IN}/{RUN}.parquet'):        # vd mir2-runs cũ chưa có rrf9_w → gộp lại từ 9 nhánh (CPU, ~1 phút)
    for f in glob.glob(f'{RUNS_IN}/*.parquet'):
        if not os.path.exists(f'{OUT}/' + os.path.basename(f)):   # cả @parents/@children: gộp theo mức như lin07
            shutil.copy(f, OUT)
    K.set({'retrieve.out_dir': OUT})
    sh(f'python -m mirret.fuse --config {K.cfg} --only {RUN}', 'fuse')
    assert os.path.exists(f'{OUT}/{RUN}.parquet'), f'fuse không tạo được {RUN} (thiếu nhánh, xem dòng "thiếu run" ở trên)'
    RUNS_IN = OUT
print('run đầu vào:', f'{RUNS_IN}/{RUN}.parquet')
def run_path(n):                                     # run trong phiên này (OUT) hoặc trong Input
    if os.path.exists(f'{OUT}/{n}.parquet'):
        return f'{OUT}/{n}.parquet'
    h = locate_all(f'runs/{n}.parquet')
    assert h, f'Không thấy runs/{n}.parquet trong /kaggle/input'
    return str(h[0] / 'runs' / f'{n}.parquet')
DEPTH = '' if TOP_N == 400 else str(TOP_N)           # TOP_N khác 400 → tên run riêng, vd lin07_cross800
sfx = '_sub' if QIDS else ''
def locate_model(name):                              # thư mục output k8c trong Input (có finetune_meta.json)
    h = sorted(glob.glob(f'{KAGGLE}/input/**/{name}/finetune_meta.json', recursive=True), key=len)
    assert h, f'Không thấy {name}/finetune_meta.json trong /kaggle/input (Add Input output k8c)'
    return os.path.dirname(h[0])
FT = {'cross': bool(CROSS_MODEL), 'qwen3i': bool(QWEN_LORA), 'qwen3': bool(QWEN_LORA)}
if CROSS_MODEL:
    K.set({'rerank.cross_model': locate_model(CROSS_MODEL)})
if QWEN_LORA:
    K.set({'rerank.qwen3_lora': locate_model(QWEN_LORA)})
    sh('python -m pip install -q peft'); sh('python -m pip uninstall -y -q torchao')   # peft mới đòi torchao > 0.16, Kaggle có 0.10, không dùng tới
assert not any(FT.get(m) for m in REUSE), 'model đã fine-tune: không dùng lại điểm cũ của phương pháp đó (bỏ khỏi REUSE)'
K.set({'rerank.ft_alpha': FT_ALPHA})
FTN = ('ft2' if any(x.endswith('synth2') for x in (CROSS_MODEL, QWEN_LORA)) else 'ft') +       (str(round(FT_ALPHA * 100)) if FT_ALPHA < 1 else '')    # vd ft50 (v1, FT_ALPHA 0,5), ft2 (model v2)
TAG = lambda m: f'{RUN}_{m}{FTN * FT.get(m, False)}{"pm" * PUBMED_ONLY}{DEPTH}{sfx}'
PM = ' --pubmed-only' if PUBMED_ONLY else ''
assert not (PUBMED_ONLY and ENSEMBLE), 'PUBMED_ONLY: tắt ENSEMBLE (ghép trên máy bằng tools/en_reorder.py)'
for m in METHODS:
    RU = REUSE.get(m) or []
    RA = ' --reuse ' + ' '.join(run_path(x) for x in ([RU] if isinstance(RU, str) else RU)) if RU else ''
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.rerank --config {K.cfg} --run {RUNS_IN}/{RUN}.parquet '
         f'--method {m} --top-n {TOP_N} --shard {g}/2 --device cuda:0 --out {OUT} --name {TAG(m)}{QARG}{RA}{PM}'
         for g in range(2)], f'rerank_{m}')
    sh(f'python -m mirret.rerank --config {K.cfg} --run {RUNS_IN}/{RUN}.parquet --method {m} --top-n {TOP_N} '
       f'--shard 0/2 --out {OUT} --name {TAG(m)}{QARG}{RA}{PM}')      # gộp 2 phần
ENS = []
if ENSEMBLE:
    assert 'cross' in METHODS and 'qwen3i' in METHODS, 'ENSEMBLE cần METHODS có cả cross và qwen3i'
    sh(f'python -m mirret.ensemble --base {RUNS_IN}/{RUN}.parquet --bge {OUT}/{TAG("cross")}.parquet '
       f'--qwen {OUT}/{TAG("qwen3i")}.parquet --top-n {TOP_N} --out {OUT} --suffix i{FTN * any(FT.values())}{DEPTH}{sfx}', 'ensemble')
    for f in glob.glob(f'{OUT}/{RUN}_ip0[68]i*'):                # ensemble.py ghi thêm hai run nội suy — không dùng
        os.remove(f)
    ENS = [f'{RUN}_ensi{FTN * any(FT.values())}{DEPTH}']

In [ ]:
# (3) Bài nộp: run gộp (không rerank) + từng run đã rerank. Run thử trên bộ nhãn (QIDS) thì không nộp.
K.set({'retrieve.out_dir': OUT})
if os.path.abspath(str(RUNS_IN)) != os.path.abspath(OUT):
    shutil.copy(f'{RUNS_IN}/{RUN}.parquet', f'{OUT}/{RUN}.parquet')
for n in NAMES[1:]:
    h = locate_all(f'runs/{n}.parquet')
    assert h, f'Không thấy runs/{n}.parquet trong /kaggle/input'
    shutil.copy(h[0] / 'runs' / f'{n}.parquet', f'{OUT}/{n}.parquet')
CPARG = ''
if CHUNK_PROBS:
    CPARG = f" --chunk-probs {run_path(CHUNK_PROBS)} --chunk-k-min {CHUNK_K_MIN}"   # run của phiên này hoặc Input
    if CHUNK_CALIB:
        hq = sorted(glob.glob(f'{KAGGLE}/input/**/{CALIB_QIDS}', recursive=True), key=len)
        assert hq, f'Không thấy {CALIB_QIDS} trong /kaggle/input'
        CPARG += f' --chunk-calib {CHUNK_CALIB} --calib-qids {hq[0]}'
for name in ([] if QIDS or PUBMED_ONLY else [RUN] * SUBMIT_RUN + NAMES[1:] + [TAG(m) for m in METHODS] + ENS):
    for mpd in MAX_PER_DOC:
        for cp in sorted({'', CPARG}):                    # CHUNK_PROBS → cả bài thường (185 đoạn) lẫn bài _cbud
            sh(f'python -m mirret.submit --config {K.cfg} --run {OUT}/{name}.parquet --k-doc {K_DOC} '
               f'--k-chunk {K_CHUNK} --doc-agg {DOC_AGG} --max-per-doc {mpd}{cp}')
report('k5')